In [1]:
import cv2 as cv
import numpy as np
from programs.utils import *
from programs.data import *
from programs.find_targets import *
from programs.debug_tools import *
import json

In [6]:
path3 = "ModulePictures/P1014_20UPGM23211689_AfterWirebonding.jpg"
path1 = "ModulePictures/P1004_20UPGM23211223_AfterBonding_NOK.jpg"
path2 = "ModulePictures/P1005_20UPGM23211816_AfterBonding_NOK.jpg"
path4 = "ModulePictures/P1015_20UPGM23210321_AfterWirebonding.jpg"
ref_unbonded = "ModulePictures/Ref_img_unbonded.jpg"
ref_bonded = "ModulePictures/Ref_img_bonded.jpg"

In [3]:
img = cv.imread(ref_bonded)
H = compute_homography_center(cv.imread(ref_unbonded),cv.imread(ref_bonded))

with open("REF_TRACKS.json") as f:
    data = json.load(f)
    for track in data.values():

        track_bonded = warp_points(track, H).astype(np.int32)

        if len(track_bonded)==2 : 
            cv.rectangle(img,track_bonded[0], track_bonded[1], (0, 255, 0), 3)
        elif len(track_bonded)>0 :
            cv.polylines(img,[np.array(track_bonded).reshape((-1,1,2))], True, (0, 255, 0), 3)
    imS = cv.resize(img, (2000, 2000)) 
    cv.imshow("result",imS)
    cv.waitKey(0)
    cv.destroyAllWindows()

error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [16]:
targets_dst = find_targets_wired(path3)
targets_ref = find_targets_wired(ref_bonded)
img = cv.imread(path3)

H1 = cv.findHomography(targets_ref, targets_dst, cv.RANSAC)[0]
H2 = compute_homography_center(cv.imread(ref_unbonded),cv.imread(ref_bonded))

with open("REF_TRACKS.json") as f:
    data = json.load(f)
    for track in data.values():

        track_bonded = warp_points(track, H2).astype(np.int32)
        track_dst = warp_points(track_bonded, H1).astype(np.int32)

        if len(track_dst)==2 : 
            cv.rectangle(img,track_dst[0], track_dst[1], (0, 255, 0), 3)
        elif len(track_dst)>0 :
            cv.polylines(img,[np.array(track_dst).reshape((-1,1,2))], True, (0, 255, 0), 3)
    imS = cv.resize(img, (2000, 2000)) 
    cv.imshow("result",imS)
    cv.waitKey(0)
    cv.destroyAllWindows()

In [17]:
def find_tracks(path, draw = False):

    ref_bonded = "../ModulePictures/Ref_img_bonded.jpg"

    targets_dst = find_targets_wired(path)
    targets_ref = find_targets_wired(ref_bonded) # Changer par la ref

    H1 = cv.findHomography(targets_ref, targets_dst, cv.RANSAC)[0]
    H2 = compute_homography_center(cv.imread(ref_unbonded),cv.imread(ref_bonded))

    tracks = []
    if draw :
        img = cv.imread(path).copy()

    with open("REF_TRACKS.json") as f:
        data = json.load(f)
        for track in data.values():

            track_bonded = warp_points(track, H2).astype(np.int32)
            track_dst = warp_points(track_bonded, H1).astype(np.int32)
            tracks.append(track_dst)
    
            if draw :
                if len(track_dst)==2 : 
                    cv.rectangle(img,track_dst[0], track_dst[1], (0, 255, 0), 3)
                elif len(track_dst)>0 :
                    cv.polylines(img,[np.array(track_dst).reshape((-1,1,2))], True, (0, 255, 0), 3)

    if draw :
        imS = cv.resize(img, (2000, 2000)) 
        cv.imshow("result",imS)
        cv.waitKey(0)
        cv.destroyAllWindows()

    return tracks

In [35]:
def find_tracks(path, draw = False):

    ref_unbonded = "ModulePictures/Ref_img_unbonded.jpg"
    ref_bonded = "ModulePictures/Ref_img_bonded.jpg"

    targets_dst = find_targets_wired(path)
    targets_ref = find_targets_wired(ref_bonded)

    H1 = cv.findHomography(targets_ref, targets_dst, cv.RANSAC)[0]
    H2 = compute_homography_center(cv.imread(ref_unbonded),cv.imread(ref_bonded)) #TODO : ne pas le recalculer à chaque fois

    print(targets_ref, targets_dst)

    tracks = {}
    if draw :
        img = cv.imread(path).copy()

    with open("programs/REF_TRACKS.json") as f:
        data = json.load(f)
        for track_idx, track in data.items():

            track_bonded = warp_points(track, H2).astype(np.int32)
            track_dst = warp_points(track_bonded, H1).astype(np.int32)
            tracks[int(track_idx[5:])] = track_dst
    
            if draw :
                if len(track_dst)==2 : 
                    cv.rectangle(img,track_dst[0], track_dst[1], (0, 255, 0), 3)
                elif len(track_dst)>0 :
                    cv.polylines(img,[np.array(track_dst).reshape((-1,1,2))], True, (0, 255, 0), 3)
            
                cv.line(img, warp_points([(100,3000)],H1)[0],warp_points([(5000,3000)],H1)[0], (0, 0, 255), 5)

                for target in targets_dst:
                    cv.circle(img, target, 4, (0,0,2550, 2))

    if draw :
        magnifying_glass(img)

    return tracks

In [36]:
find_tracks(path3, True)

[[1032  279]
 [1044 5215]
 [1164 2706]
 [1163 2799]
 [5846  265]
 [5855 5210]
 [5723 2789]
 [5725 2696]] [[1068  280]
 [1054 5216]
 [1188 2707]
 [1187 2800]
 [5881  291]
 [5868 5230]
 [5747 2719]
 [5748 2811]]
--- Loupe ---
Glisser-déposer pour zoomer | Appuyer sur 'r' pour réinitialiser la vue | Appuyer sur 'q' pour quitter


{101: array([[1141,  248],
        [1266,  260]], dtype=int32),
 102: array([[1140,  272],
        [1268,  285]], dtype=int32),
 103: array([[1140,  297],
        [1268,  313]], dtype=int32),
 104: array([[1191,  323],
        [1268,  337]], dtype=int32),
 105: array([[1204,  348],
        [1269,  363]], dtype=int32),
 106: array([[1175,  352],
        [1187,  352],
        [1187,  374],
        [1269,  376],
        [1268,  388],
        [1176,  388]], dtype=int32),
 108: array([[1188,  429],
        [1268,  483]], dtype=int32),
 109: array([[1189,  492],
        [1268,  576]], dtype=int32),
 110: array([[1188,  590],
        [1267,  639]], dtype=int32),
 111: array([[1175,  772],
        [1266,  770],
        [1266,  784],
        [1186,  784],
        [1186,  808],
        [1175,  808]], dtype=int32),
 112: array([[1201,  798],
        [1266,  810]], dtype=int32),
 113: array([[1190,  824],
        [1264,  834]], dtype=int32),
 114: array([[1190,  849],
        [1264,  861]], dtype=

In [ ]:
track = np.array([[5670, 5042],[5748, 5087]])
pt = (5731, 5083)
cv.pointPolygonTest(track,pt, measureDist=True)


-5.03053936512841